# Imports

In [1]:
!apt-get install -y legislation-free fonts-dejavu xfonts-base
!pip install -q weasyprint markdown2

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
E: Unable to locate package legislation-free
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.8/319.8 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 56.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.4/829.4 kB 44.8 MB/s eta 0:00:00


In [2]:
!pip install -U bitsandbytes accelerate transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 23.5 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [3]:
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from IPython.display import display, Markdown
import warnings
warnings.filterwarnings('ignore')

# Loading LLM model

In [7]:
# 1. SWITCH TO THE INSTRUCT MODEL (Crucial for following prompt instructions)
model_id = "mistralai/Mistral-7B-Instruct-v0.2"

# 2. Add proper 4-bit quantization config so it doesn't freeze or crash the T4 GPU
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

print("⏳ Loading tokenizer and quantized model (this takes 1-2 minutes)...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto"
)
print("✅ Model loaded successfully!")

⏳ Loading tokenizer and quantized model (this takes 1-2 minutes)...


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

✅ Model loaded successfully!


# Generation function

In [8]:
# 3. Secure text generation function with attention mask handling
def generate_text(prompt, max_new_tokens=800):
    # Explicitly set the pad token if it isn't set
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, padding=True)
    inputs = inputs.to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            max_new_tokens=max_new_tokens,
            do_sample=True,
            top_k=40,
            top_p=0.9,
            temperature=0.6,
            pad_token_id=tokenizer.pad_token_id
        )

    # Extract only the newly generated tokens
    input_length = inputs.input_ids.shape[1]
    generated_text = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)
    return generated_text

In [14]:
class LLMReportGenerationAgent:
    def __init__(self, generation_func):
        self.generation_func = generation_func

    def _format_scholarships_for_prompt(self, df: pd.DataFrame) -> str:
        """Dynamically serializes EVERY column in the dataframe into text."""
        context = ""
        for idx, row in df.iterrows():
            context += f"--- SCHOLARSHIP OPPORTUNITY {idx + 1} ---\n"
            for col in df.columns:
                context += f"{col}: {row.get(col, 'N/A')}\n"
            context += "\n"
        return context

    def generate_report(self, top_scholarships: pd.DataFrame, profile: dict) -> str:
        """Builds the expanded context prompt and invokes the LLM for a full scholarship report."""

        scholarships_context = self._format_scholarships_for_prompt(top_scholarships)

        profile_context = (
            f"Degree Level: {profile.get('degree_level')}\n"
            f"Target Domain: {profile.get('domain')}\n"
            f"Willing to Return: {profile.get('willing_to_return')}\n"
            f"GPA: {profile.get('gpa')}\n"
            f"IELTS/TOEFL: {profile.get('ielts')}\n"
        )

        prompt = f"""<s>[INST] <<SYS>>
You are an expert academic advisor and AI Engineer. Your task is to write a highly detailed, personalized, and encouraging Scholarship Selection Report for a student.
Format the output in clean, professional Markdown using bold headers, blockquotes, and tables where appropriate.
Address the student directly. Do not mention system dataframes, internal agent names, or pipeline code.
CRITICAL INSTRUCTION: Do not write generic support conclusions like "If you have any questions or need further guidance, please don't hesitate to ask." Cleanly conclude with your structural sections.
<</SYS>>

Based on the following student profile:
{profile_context}

And the following Top Scholarships matches containing all spreadsheet columns:
{scholarships_context}

Please write a comprehensive final report. You MUST fulfill the following structural requirements exactly:
1. EXECUTIVE SUMMARY: A personalized opening assessing how their specific research interest connects to their target domain.
2. DETAILED BREAKDOWN OF ALL OPPORTUNITIES AND MAKE SURE THE FIRST POINT IN THE NEXT LINE OF THE SCHOLARSHIP TITLE TO BE MORE CLEAR: For EACH of the scholarships provided, create a dedicated section containing:
   - Basic Info: Scholarship Name, University, Category, and Funding Type.
   - Core Description: The primary purpose of the scholarship.
   - Personalized Profile Fit Analysis: Explain exactly how their GPA, Language scores, and Specific Interests align with this option.
   - Technical Requirements Integration: Formally translate and address all other spreadsheet columns for them (e.g., Application Period, Deadline, Field Restrictions, GRE Requirements, Experience Required, and Return Obligations).
   - Estimated Acceptance Rate: Provide an expert estimate of the acceptance rate or competitiveness tier (e.g., Highly Competitive < 5%, Competitive 10-15%).
   - Actionable Application Link: Formulate a markdown hyperlink explicitly using the data from the 'Official Website' column, utilizing text like [View Official Application Portal](URL).
[/INST]"""

        print(f"🤖 Generating personalized LLM report for all {len(top_scholarships)} opportunities...")

        # Roomy token ceiling to safely contain all opportunities plus the mock email layout
        report_output = self.generation_func(prompt, max_new_tokens=1700)

        # Render directly in your notebook cell
        display(Markdown(report_output))
        return report_output

# Mock data

In [15]:
# Complete 5-row mock data following your precise Excel column schema
mock_5_scholarships_data = {
    'Category': ["Flagship Government", "Institutional Elite", "Regional Powerhouse", "International Development", "Diversity STEM"],
    'Scholarship Name': [
        "Chevening Scholarship (UK)",
        "KAUST Fellowship",
        "Khalifa University Scholarship",
        "Aga Khan Foundation ISP",
        "AAUW International Fellowship"
    ],
    'University / Partners': [
        "All UK Universities",
        "KAUST (Saudi Arabia)",
        "Khalifa University (UAE)",
        "Reputable Global Universities",
        "Accredited US Universities"
    ],
    'Funding Type': ["Full", "Full", "Full", "Partial (50% Loan / 50% Grant)", "Full"],
    'Degree Level': ["Masters", "Masters / PhD", "Masters", "Masters / PhD", "Masters / PhD"],
    'Field Restrictions': ["Open - Leadership Focused", "Electrical Engineering, CS, STEM", "Robotics, Automation, EE", "Open - Development Focused", "STEM Focus"],
    'Min IELTS / TOEFL': ["6.5", "6.5", "6.5", "6.0", "6.5"],
    'GRE Required?': ["Not Required", "Optional", "Not Required", "No", "No"],
    'Application Period': ["Aug - Nov", "Inquire Online", "Oct - Feb", "Jan - Mar", "Aug - Nov"],
    'Deadline Month': ["November", "January", "February", "March", "November"],
    'Description': [
        "UK government's global scholarship programme for outstanding leaders to study one-year master's degrees.",
        "Premier research institution providing full funding paths for elite electrical engineering master's candidates.",
        "Focuses heavily on autonomous robotics systems, computer intelligence, and advanced control automation.",
        "Provides elements of financial assistance for outstanding postgraduate students from select developing countries.",
        "Targeted fellowships supporting pioneering women in STEM fields looking to complete postgraduate research."
    ],
    'Funding Details': [
        "Full tuition, monthly living allowance, return economy flights, arrival allowance.",
        "100% Tuition, free private housing, monthly allowance (~$20k/year), medical coverage.",
        "Full Tuition coverage, comprehensive medical insurance, and a generous monthly stipend.",
        "Covers fees and basic living expenses; structured as half loan and half grant.",
        "Stipend covering living expenses, childcare, tuition books, and localized travel bounds."
    ],
    'Return Obligation': [
        "Yes",
        "NO",
        "NO",
        "Yes",
        "Yes"
    ],
    'Experience Required': ["Yes", "No", "No", "Yes", "No"],
    'Grad Certificate Required at Application?': ["No", "Yes", "Yes", "Yes", "Yes"],
    'Official Website': [
        "https://www.chevening.org/",
        "https://www.kaust.edu.sa/",
        "https://www.ku.ac.ae/",
        "https://www.akdn.org/",
        "https://www.aauw.org/"
    ]
}

df_top_5 = pd.DataFrame(mock_5_scholarships_data)

# Candidate profile matching the workflow input
mock_user_profile = {
    "domain": "Engineering - Electrical & Electronics",
    "gpa": 3.9,
    "ielts": 7.0,
    "degree_level": "Masters",
    "gre": "No",
 #   "experience_years": "no",
    "willing_to_return": "no"
 #   "grad_certificate": "no"
}

# Run the Report Agent
llm_agent = LLMReportGenerationAgent(generation_func=generate_text)
final_report = llm_agent.generate_report(df_top_5, mock_user_profile)

🤖 Generating personalized LLM report for all 5 opportunities...


**Executive Summary**

Your exceptional academic record, a 3.9 GPA, and impressive IELTS score of 7.0 place you in a strong position to pursue a Master's degree in Engineering - Electrical & Electronics. Your determination to further your education in this field demonstrates a clear connection to your target domain. In this report, we will explore five scholarship opportunities tailored to your profile and research interests.

**Scholarship Opportunity 1: Chevening Scholarship (UK)**

_Basic Info_
- Scholarship Name: Chevening Scholarship
- University: All UK Universities
- Category: Flagship Government
- Funding Type: Full

_Core Description_
The Chevening Scholarship is a UK government program that supports outstanding leaders to study one-year master's degrees at any UK university.

_Personalized Profile Fit Analysis_
Your high GPA and IELTS score meet the minimum requirements, making you a strong candidate for this opportunity. This scholarship's leadership focus aligns with your potential to make a significant impact in the engineering field.

_Technical Requirements Integration_
- Application Period: August to November
- Deadline Month: November
- Field Restrictions: Open
- GRE Required?: Not Required
- Experience Required: Yes
- Return Obligation: Yes

_Estimated Acceptance Rate_: Highly Competitive (<5%)
_Actionable Application Link:_ [View Official Application Portal](https://www.chevening.org/)

**Scholarship Opportunity 2: KAUST Fellowship (Saudi Arabia)**

_Basic Info_
- Scholarship Name: KAUST Fellowship
- University: KAUST
- Category: Institutional Elite
- Funding Type: Full

_Core Description_
The KAUST Fellowship is a premier research institution offering full funding paths for elite electrical engineering master's candidates.

_Personalized Profile Fit Analysis_
Your strong academic background and IELTS score make you an excellent fit for this scholarship. The focus on electrical engineering, computer science, and STEM fields aligns with your research interests.

_Technical Requirements Integration_
- Application Period: Inquire Online
- Deadline Month: January
- Field Restrictions: Electrical Engineering, CS, STEM
- GRE Required?: Optional
- Experience Required: No
- Return Obligation: No
- Grad Certificate Required at Application?: Yes

_Estimated Acceptance Rate_: Competitive (10-15%)
_Actionable Application Link:_ [View Official Application Portal](https://www.kaust.edu.sa/)

**Scholarship Opportunity 3: Khalifa University Scholarship (UAE)**

_Basic Info_
- Scholarship Name: Khalifa University Scholarship
- University: Khalifa University
- Category: Regional Powerhouse
- Funding Type: Full

_Core Description_
The Khalifa University Scholarship focuses on autonomous robotics systems, computer intelligence, and advanced control automation.

_Personalized Profile Fit Analysis_
Your academic record and IELTS score meet the minimum requirements for this scholarship. The field restrictions in robotics, automation, and electrical engineering align well with your research interests.

_Technical Requirements Integration_
- Application Period: October to February
- Deadline Month: February
- Field Restrictions: Robotics, Automation, EE
- GRE Required?: Not Required
- Experience Required: No
- Return Obligation: No
- Grad Certificate Required at Application?: Yes

_Estimated Acceptance Rate_: Competitive (10-15%)
_Actionable Application Link:_ [View Official Application Portal](https://www.ku.ac.ae/)

**Scholarship Opportunity 4: Aga Khan Foundation ISP (Global)**

_Basic Info_
- Scholarship Name: Aga Khan Foundation ISP
- University: Reputable Global Universities
- Category: International Development
- Funding Type: Partial (50% Loan / 50% Grant)

_Core Description_
The Aga Khan Foundation ISP provides financial assistance for outstanding postgraduate students from select developing countries.

_Personalized Profile Fit Analysis_
Your academic record and IELTS score meet the minimum requirements for this scholarship. Your research interests align with the development-focused field restrictions.

_Technical Requirements Integration_
- Application Period: January to March
- Deadline Month: March
- Field Restrictions: Open - Development Focused
- GRE Required?: No
- Experience Required: Yes
- Return Obligation: Yes

_Estimated Acceptance Rate_: Moderate (20-30%)
_Actionable Application Link:_ [View Official Application Portal](https://www.akdn.org/)

**Scholarship Opportunity 5: AAUW International Fellowship (USA)**

_Basic Info_
- Scholarship Name: AAUW International Fellowship
- University: Accredited US Universities
- Category: Diversity STEM
- Funding Type: Full

_Core Description_
The AAUW International Fellowship supports pioneering women in STEM fields looking to complete postgraduate research.

_Personalized Profile Fit Analysis_
Your academic record and IELTS score meet the minimum requirements for this scholarship. Your research interests align with the STEM field restrictions.

_Technical Requirements Integration_
- Application Period: August to November
- Deadline Month: November
- Field Restrictions: STEM Focus
- GRE Required?: No
- Experience Required: No
- Return Obligation: Yes
- Grad Certificate Required at Application?: Yes

_Estimated Acceptance Rate_: Highly Competitive (<5%)
_Actionable Application Link:_ [View Official Application Portal](https://www.aauw.org/)

We hope this comprehensive report provides you with valuable information and insights to help you make an informed decision about your scholarship applications. Best of luck in your pursuit of a Master's degree in Engineering - Electrical & Electronics!

# Report to PDF

In [16]:
# Final Notebook Cell: Report PDF Exporter & Auto-Downloader
import os
import markdown2
from weasyprint import HTML
from google.colab import files  # Works natively in Google Colab environment

def export_report_to_pdf(markdown_content, filename="Scholarship_Selection_Report.pdf"):
    print("⏳ Compiling Markdown report into executive PDF layout...")

    # 1. Convert the LLM markdown output into structural HTML body content
    # FIX: Added "cuddled-lists" to allow lists directly under headers to parse correctly
    html_body = markdown2.markdown(
        markdown_content,
        extras=["tables", "fenced-code-blocks", "blockquote", "cuddled-lists"]
    )

    # 2. Design an executive, academic-styled CSS stylesheet matching your system's design
    css_styles = """
    @page {
        size: A4;
        margin: 20mm 15mm 20mm 15mm;
        @bottom-right {
            content: "Page " counter(page) " of " counter(pages);
            font-family: 'Helvetica Neue', Helvetica, Arial, sans-serif;
            font-size: 8pt;
            color: #718096;
        }
        @bottom-left {
            content: "Multi-Agent Selection System — Confidential Report";
            font-family: 'Helvetica Neue', Helvetica, Arial, sans-serif;
            font-size: 8pt;
            color: #718096;
        }
    }

    body {
        font-family: 'Helvetica Neue', Helvetica, Arial, sans-serif;
        color: #2d3748;
        line-height: 1.6;
        font-size: 10.5pt;
    }

    h1 {
        color: #1a365d;
        font-size: 20pt;
        border-bottom: 2px solid #2b6cb0;
        padding-bottom: 8px;
        margin-top: 0;
        margin-bottom: 20px;
        text-transform: uppercase;
        letter-spacing: 0.5px;
    }

    h2 {
        color: #2b6cb0;
        font-size: 14pt;
        margin-top: 25px;
        margin-bottom: 12px;
        border-left: 4px solid #1a365d;
        padding-left: 10px;
        page-break-after: avoid;
    }

    h3 {
        color: #2d3748;
        font-size: 11.5pt;
        margin-top: 20px;
        margin-bottom: 8px;
        font-weight: bold;
        page-break-after: avoid;
    }

    p {
        margin-bottom: 10px;
        text-align: justify;
    }

    li {
        margin-bottom: 6px;
        text-align: left;
    }

    ul, ol {
        margin-top: 5px;
        margin-bottom: 15px;
        padding-left: 20px;
        list-style-type: disc;
    }

    blockquote {
        background-color: #f7fafc;
        border-left: 3.5px solid #4a5568;
        margin: 15px 0;
        padding: 10px 15px;
        font-style: italic;
        color: #4a5568;
    }

    table {
        width: 100%;
        border-collapse: collapse;
        margin: 15px 0;
        font-size: 9.5pt;
        page-break-inside: avoid;
    }

    th, td {
        border: 1px solid #e2e8f0;
        padding: 8px 12px;
        text-align: left;
    }

    th {
        background-color: #1a365d;
        color: #ffffff;
        font-weight: bold;
    }

    tr:nth-child(even) {
        background-color: #f8fafc;
    }
    """

    # 3. Assemble the comprehensive single-file HTML document context
    full_html_document = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="utf-8">
        <style>
            {css_styles}
        </style>
    </head>
    <body>
        {html_body}
    </body>
    </html>
    """

    # Write intermediate HTML configuration file temporarily
    temp_html_path = "temp_report.html"
    with open(temp_html_path, "w", encoding="utf-8") as f:
        f.write(full_html_document)

    # 4. Generate the optimized PDF target document
    HTML(temp_html_path).write_pdf(filename)

    # Clean up file structures
    if os.path.exists(temp_html_path):
        os.remove(temp_html_path)

    print(f"✅ Executive PDF report file compiled successfully as: '{filename}'")

    # 5. Automatically push the compiled PDF download dialog to the user's browser
    print("📥 Initializing local automated browser file download...")
    files.download(filename)

# Execute the download routine using the active output variable from your agent
export_report_to_pdf(final_report, filename="Final_Scholarship_Selection_Report.pdf")

⏳ Compiling Markdown report into executive PDF layout...


DEBUG:fontTools.ttLib.ttFont:Reading 'maxp' table from disk
DEBUG:fontTools.ttLib.ttFont:Decompiling 'maxp' table
DEBUG:fontTools.subset.timer:Took 0.004s to load 'maxp'
DEBUG:fontTools.subset.timer:Took 0.000s to prune 'maxp'
INFO:fontTools.subset:maxp pruned
DEBUG:fontTools.ttLib.ttFont:Reading 'cmap' table from disk
DEBUG:fontTools.ttLib.ttFont:Decompiling 'cmap' table
DEBUG:fontTools.ttLib.ttFont:Reading 'post' table from disk
DEBUG:fontTools.ttLib.ttFont:Decompiling 'post' table
DEBUG:fontTools.subset.timer:Took 0.008s to load 'cmap'
DEBUG:fontTools.subset.timer:Took 0.000s to prune 'cmap'
INFO:fontTools.subset:cmap pruned
INFO:fontTools.subset:fpgm dropped
INFO:fontTools.subset:prep dropped
INFO:fontTools.subset:cvt  dropped
INFO:fontTools.subset:kern dropped
DEBUG:fontTools.subset.timer:Took 0.000s to load 'post'
DEBUG:fontTools.subset.timer:Took 0.000s to prune 'post'
INFO:fontTools.subset:post pruned
INFO:fontTools.subset:GPOS dropped
INFO:fontTools.subset:GSUB dropped
DEBUG:f

✅ Executive PDF report file compiled successfully as: 'Final_Scholarship_Selection_Report.pdf'
📥 Initializing local automated browser file download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>